# Lab 09-04 · Generation, Recombination and the Continuity Equation

Companion to chapter [09-04](../../chapters/09-04-generation-recombination-continuity.md). Handout: [README.md](README.md).

Non-equilibrium carriers computed numerically: a trap level with its four capture and emission processes integrated in time, diffusion profiles as boundary-value problems, and a wafer's photoconductive decay simulated with surface recombination.

In [1]:
import math
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

KT = 8.617333262e-5 * 300
NI = 1e10
VN, VP = 2.3e7, 1.65e7
mup = lambda N: 48 + 402 / (1 + (N / 6.3e16) ** 0.76)
print("ready")

ready


## Part 1 — A trap level in time

Electrons, holes and trapped electrons in n-type silicon ($N_D$ = 10¹⁶ cm⁻³) with gold-like traps ($N_t$ = 10¹⁵ cm⁻³ is large, so use 10¹³ to keep low injection clean). Start with an excess of 10¹² cm⁻³ and integrate the four processes.

**🔮 Predict:** the decay time of the excess holes. Is it $\tau_{p0}$, $\tau_{n0}$ or something else?

In [2]:
ND, Nt = 1e16, 1e13
sn, sp = 8e-17, 9.0e-15
cn, cp = sn * VN, sp * VP
EtEi = 0.0158
n1, p1 = NI * math.exp(EtEi / KT), NI * math.exp(-EtEi / KT)
n0 = ND; p0 = NI**2 / ND
ft0 = (cn * n0 + cp * p1) / (cn * (n0 + n1) + cp * (p0 + p1))       # equilibrium occupation
def rhs(t, y):
    n, p, ft = y
    rn = cn * n * Nt * (1 - ft) - cn * n1 * Nt * ft                  # net electron capture
    rp = cp * p * Nt * ft - cp * p1 * Nt * (1 - ft)                  # net hole capture
    return [-rn, -rp, (rn - rp) / Nt]
d0 = 1e12
sol = solve_ivp(rhs, (0, 5e-6), [n0 + d0, p0 + d0, ft0], method="LSODA", rtol=1e-9, atol=[1e3, 1e-3, 1e-15],
                dense_output=True)
t = np.linspace(2e-7, 1.5e-6, 200)
dp = sol.sol(t)[1] - p0
tau_fit = -1 / np.polyfit(t, np.log(dp), 1)[0]
tau_p0 = 1 / (cp * Nt)
print(f"fitted decay {tau_fit*1e9:.1f} ns; tau_p0 = {tau_p0*1e9:.1f} ns; tau_n0 = {1/(cn*Nt)*1e9:.1f} ns")
assert abs(tau_fit / tau_p0 - 1) < 0.02

fitted decay 675.3 ns; tau_p0 = 673.4 ns; tau_n0 = 54347.8 ns


## Part 2 — Trap energy and generation

In a depletion region ($n, p \ll n_i$) the generation rate is $n_i/[\tau_0(e^{(E_t-E_i)/kT} + e^{(E_i-E_t)/kT})]$. Sweep $E_t$.

In [3]:
E = np.linspace(-0.5, 0.5, 401)
G = NI / (1e-6 * (np.exp(E / KT) + np.exp(-E / KT)))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.semilogy(E, G); ax.set_xlabel("E_t - E_i (eV)"); ax.set_ylabel("generation rate (cm^-3 s^-1)")
fig.tight_layout(); plt.close(fig)
print(f"peak {G.max():.3e} at E_t = E_i; 0.3 eV off: {NI/(1e-6*2*math.cosh(0.3/KT)):.3e}")
assert abs(G.max() - 5e15) < 1e10

peak 5.000e+15 at E_t = E_i; 0.3 eV off: 9.125e+10


## Part 3 — Diffusion profiles as boundary-value problems

Solve $D_p\Delta p'' - \Delta p/\tau_p = 0$ by finite differences for a long region and for a short region ending at a contact, and compute the injected current from the slope.

In [4]:
def bvp(L, W, n=4001):
    x = np.linspace(0, W, n); h = x[1] - x[0]
    A = np.zeros((n, n)); b = np.zeros(n)
    A[0, 0] = A[-1, -1] = 1; b[0] = 1
    for i in range(1, n - 1):
        A[i, i-1] = A[i, i+1] = 1 / h**2; A[i, i] = -2 / h**2 - 1 / L**2
    y = np.linalg.solve(A, b)
    return x, y
Dp = KT * mup(1e15); Lp = math.sqrt(Dp * 1e-6)
for W in (10 * Lp, 5e-4):
    x, y = bvp(Lp, W)
    slope = -(y[1] - y[0]) / (x[1] - x[0])
    J = 1.602176634e-19 * Dp * 1e14 * slope
    print(f"W = {W*1e4:.1f} um: J = {J*1e3:.1f} mA/cm^2")
x, y = bvp(Lp, 5e-4)
J_short = 1.602176634e-19 * Dp * 1e14 * (-(y[1] - y[0]) / (x[1] - x[0]))
J_exact = 1.602176634e-19 * Dp * 1e14 / (Lp * math.tanh(5e-4 / Lp))
print(f"exact (coth) result: {J_exact*1e3:.1f} mA/cm^2; linear approximation qD dp/W: {1.602176634e-19*Dp*1e14/5e-4*1e3:.1f}")
assert abs(J_short / J_exact - 1) < 2e-3

W = 334.8 um: J = 53.6 mA/cm^2


W = 5.0 um: J = 361.7 mA/cm^2


exact (coth) result: 361.7 mA/cm^2; linear approximation qD dp/W: 359.1


## Part 4 — What a lifetime tester sees (the chapter's guided exercise)

A 300 µm wafer, bulk lifetime 1 ms, both surfaces with recombination velocity $S$. Simulate the decay after a uniform light pulse with the diffusion equation and surface boundary conditions, and fit the late-time decay.

**🔮 Predict:** with $S$ = 10⁴ cm/s, is the measured lifetime near 1 ms, 100 µs or 10 µs?

In [5]:
def pcd_tau(S, W=0.03, taub=1e-3, D=None, n=301):
    # effective lifetime from a simulated decay; fit the late part, where only the slowest mode survives
    D = D or KT * mup(1e15)
    x = np.linspace(0, W, n); h = x[1] - x[0]
    # method of lines: interior diffusion, Robin boundaries D dp/dx = S p (x = 0), -D dp/dx = S p (x = W)
    def rhs(t, p):
        d = np.empty_like(p)
        d[1:-1] = D * (p[2:] - 2 * p[1:-1] + p[:-2]) / h**2
        d[0] = D * 2 * (p[1] - p[0] - h * S / D * p[0]) / h**2
        d[-1] = D * 2 * (p[-2] - p[-1] - h * S / D * p[-1]) / h**2
        return d - p / taub
    t_end = 0.5 * min(taub, W * W / D)                # a few decay times of the slowest mode
    sol = solve_ivp(rhs, (0, t_end), np.ones(n), method="BDF", t_eval=np.linspace(0.5 * t_end, t_end, 50),
                    rtol=1e-8, atol=1e-14)
    total = sol.y.sum(axis=0)
    return -1 / np.polyfit(sol.t, np.log(total), 1)[0]

def exact_tau(S, W=0.03, taub=1e-3, D=None):
    D = D or KT * mup(1e15)
    y = brentq(lambda y: y * math.tan(y) - S * W / (2 * D), 1e-9, math.pi / 2 - 1e-12)
    a = 2 * y / W
    return 1 / (1 / taub + D * a * a)

for S in (10.0, 1e4):
    print(f"S = {S:g} cm/s: simulated {pcd_tau(S)*1e6:.1f} us, exact mode {exact_tau(S)*1e6:.2f} us, "
          f"weak-surface formula {1/(1/1e-3 + 2*S/0.03)*1e6:.1f} us")
assert abs(pcd_tau(1e4) / exact_tau(1e4) - 1) < 0.03

S = 10 cm/s: simulated 601.1 us, exact mode 601.07 us, weak-surface formula 600.0 us


S = 10000 cm/s: simulated 9.3 us, exact mode 9.31 us, weak-surface formula 1.5 us


## Wrap-up

- With a single mid-gap trap in n-type silicon, the excess decays with $\tau_{p0}$: minority capture is the bottleneck.
- Generation needs deep levels: moving a trap 0.3 eV from mid-gap cuts generation by more than four orders of magnitude.
- Short regions carry larger diffusion currents with almost no recombination.
- An unpassivated wafer's measured lifetime reflects its surfaces, not its bulk: at $S$ = 10⁴ cm/s a 1 ms wafer measures about 9 µs, and the weak-surface formula fails badly.